In [2]:
#
# python run.py --input_file shared/orlov.mp4 --output_file res.mp4
#
import sys
import argparse
import subprocess
import os
# voice clonning
from replica import utils as replica
### TODO: убрать лишние зависимости
from bark.generation import SAMPLE_RATE
from df.enhance import enhance, load_audio, save_audio

if not os.path.exists("temp"):
    os.makedirs("temp")

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2023-07-09 19:32:58 | INFO | fairseq.tasks.text_to_speech | Please install tensorboardX: pip install tensorboardX
/usr/local/lib/python3.10/site-packages/whisper/timing.py:58: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def backtrace(trace: np.ndarray):


In [ ]:
input_file = 'shared/orlov.mp4'
output_file = 'temp/res.mp4'
device = "cuda" # or "cpu"

In [46]:
from IPython.display import Audio
import librosa
df_model, df_state = replica.voice_cleaning_setup()
replica.clean_audio(df_model, df_state, "temp/input_audio.wav", "temp/clean_audio.wav")
cmd = f"ffmpeg -y -i temp/clean_audio.wav -af silenceremove=stop_periods=-1:stop_duration=1:stop_threshold=-13dB temp/input_audio_wo_silence.wav"
subprocess.run(cmd.split())
print(librosa.get_duration(filename="temp/input_audio_wo_silence.wav"))
Audio('temp/input_audio_wo_silence.wav', rate=SAMPLE_RATE)

2023-07-09 13:56:54 | INFO     | DF | Loading model settings of DeepFilterNet3
2023-07-09 13:56:54 | INFO     | DF | Using DeepFilterNet3 model at /root/.cache/DeepFilterNet/DeepFilterNet3
2023-07-09 13:56:54 | INFO     | DF | Initializing model `deepfilternet3`
2023-07-09 13:56:54 | INFO     | DF | Found checkpoint /root/.cache/DeepFilterNet/DeepFilterNet3/checkpoints/model_120.ckpt.best with epoch 120
2023-07-09 13:56:54 | INFO     | DF | Running on device cuda:0
2023-07-09 13:56:54 | INFO     | DF | Model loaded
4.0108958333333335


/usr/local/lib/python3.10/site-packages/torchaudio/functional/functional.py:1458: UserWarning: "sinc_interpolation" resampling method name is being deprecated and replaced by "sinc_interp_hann" in the next release. The default behavior remains unchanged.
  warnings.warn(
ffmpeg version 5.1.3-1 Copyright (c) 2000-2022 the FFmpeg developers
  built with gcc 12 (Debian 12.2.0-14)
  configuration: --prefix=/usr --extra-version=1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-li

In [ ]:
### TODO: добавить аргумент verbose и выводить логи поэтапно
cmd = f"ffmpeg -y -i {input_file} -ss 0 -t 10 temp/input_audio.wav"
subprocess.run(cmd.split())

### TODO: пройтись по записи и проанализировать качество аудио и количество голосов/роли
### определить правильность выбранного голоса (голос для клонирования должен быть один и достаточно чистый)
### если несколько человек, то по тембру выбрать голос каждого и склонировать каждого с отметкой его тембра для соответствующего синтеза
### по спектру смотреть присутствуют ли в аудио отрезке другие звуки, если есть только голосовые частоты, то брать для клонирования. проходиться окном по аудио и искать отрезок с голосом и минимумом посторонних звуков
df_model, df_state = replica.voice_cleaning_setup()
replica.clean_audio(df_model, df_state, "temp/input_audio.wav", "temp/clean_audio.wav")

codec_model = replica.voice_clonning_setup_bark(device)
tokenizer_file = replica.voice_clonning_download_hubert("eng")
hubert_model = replica.voice_clonning_setup_hubert(device)
tokenizer_model = replica.voice_clonning_setup_tokenizer(device, tokenizer_file)
replica.clone_voice(device, hubert_model, tokenizer_model, codec_model, "temp/clean_audio.wav", "temp/voice_clone.npz")

In [ ]:
whisper_model = replica.transcribe_audio_setup("small")
text_transcribed = replica.transcribe_audio(whisper_model, "temp/clean_audio.wav")

translate_model = replica.translate_text_setup(device, "ru-en")
text_translated = replica.translate_text(translate_model, text_transcribed)

In [ ]:
print(text_transcribed)
print(text_translated)

In [ ]:
del codec_model, hubert_model, tokenizer_model

In [ ]:
#text_translated = 'Hello. My name is Sergey Orlov. And this is my best show in the Internet'

In [ ]:
del whisper_model, translate_model

In [ ]:
from IPython.display import Audio
replica.voice_synthesis_setup()

In [ ]:
resemblyzer_encoder = replica.resemblyzer_setup()

In [ ]:
Audio('temp/clean_audio.wav', rate=SAMPLE_RATE)

In [ ]:
from scipy.io.wavfile import write as write_wav
audio_array = replica.synthesize_voice(text_translated, "temp/voice_clone.npz", "simple") #bark-with-voice-clone/bark/assets/prompts/ru_speaker_0.npz temp/voice_clone.npz
write_wav("temp/voice_synt_noise.wav", SAMPLE_RATE, audio_array)
Audio('temp/voice_synt_noise.wav', rate=SAMPLE_RATE)

In [ ]:
fpath = Path("temp/clean_audio.wav")
wav = preprocess_wav(fpath)
embeds_a = resemblyzer_encoder.embed_utterance(wav)
np.set_printoptions(precision=3, suppress=True)

fpath = Path("temp/voice_synt_noise.wav")
wav = preprocess_wav(fpath)
embeds_b = resemblyzer_encoder.embed_utterance(wav)
np.set_printoptions(precision=3, suppress=True)

sim = np.inner(embeds_a, embeds_b)
print(sim)

In [ ]:
проходить по всему аудио и составлять статистику, количество присутствующих голосов и где звучит их чистый голос без шумов по спектру и других голосов
проходить по чистым отрывкам, все их пытаться клонировать и проверять на похожесть после генерации
удалять паузы в аудио для клонирования
похожесть подготовленного npz делать в цикле из (20), выбирать по максимальному скору, скор считать как медиану между всеми sim сгенеренных аудио

In [ ]:
speech_list = replica.synthesize_voice_list(text_translated, "temp/voice_clone.npz", resemblyzer_encoder, "simple", "temp/clean_audio.wav", 10)
Audio(speech_list[0][2], rate=SAMPLE_RATE)

In [ ]:
whisper_model = replica.transcribe_audio_setup("small")
best_speech_file = find_best_sample(text_translated, speech_list, whisper_model)
del whisper_model

In [ ]:
replica.clean_audio(df_model, df_state, best_speech_file, "temp/voice_synt.wav")
replica.video_synchronization_setup()
replica.sync_video(input_file, "temp/voice_synt.wav", output_file)

In [ ]:
# Display video.
from IPython.display import HTML
from base64 import b64encode

def show_video(path):
    mp4 = open(path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f"""<video width=600 controls><source src='{data_url}'></video>""")

# Preview the trimmed video
show_video(output_file)